# MAS Vocabulary and Architecture Overview

Multi-agent systems (MAS) introduce many terms — **agent**, **orchestrator**, **supervisor**, **handoff**, **termination** — that different frameworks name differently. Without a shared vocabulary, design discussions get confusing fast.

This notebook gives you:

- **Precise definitions** for core MAS terms.
- A **layered architecture** diagram showing where each component lives.
- A **framework mapping** (LangGraph, AutoGen, CrewAI, MCP) so you can translate concepts across tools.
- A **working ShopCo support system** with a supervisor, specialist workers, handoffs, and termination — all in plain Python.

Everything is self-contained. Run the cells to see vocabulary in action, not just on slides.

In [ ]:
"""
ShopCo Customer Support — minimal backend for MAS vocabulary demo.
"""

import json
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


ORDERS = {
    "ORD-1001": {"customer": "alice@shop.com", "status": "shipped", "amount": 1299.00},
    "ORD-1002": {"customer": "bob@shop.com", "status": "processing", "amount": 449.00},
}

POLICIES = [
    {"id": "ret-02", "text": "Returns within 30 days if unused."},
    {"id": "cancel-03", "text": "Cancellations allowed only before shipment."},
]

print(f"ShopCo backend ready — {len(ORDERS)} orders, {len(POLICIES)} policy docs")

---

## 1. Glossary — Core Terms

| Term | Definition |
|------|------------|
| **Agent** | An LLM-powered entity that **perceives** (reads messages and observations), **decides** (plans tool use or text), and **acts** (calls tools or responds) in a loop |
| **Tool** | A named, schema-defined function the model can invoke — your agent-computer interface (ACI) |
| **Observation** | The string or structured payload returned after tool execution — input to the next model turn |
| **Orchestrator** | Control logic that runs the agent loop: routes state, invokes model/tools, enforces limits |
| **Supervisor** | A specialized orchestrator agent that **delegates** to worker agents and merges results |
| **Handoff** | Passing control and context from one agent to another with an explicit message contract |
| **Termination** | Condition that ends the loop: final answer, max iterations, guardrail block, or human stop |

These seven terms appear in almost every MAS design doc. Learn them once; they transfer across frameworks.

---

## 2. Agent vs Tool vs Workflow

```
Workflow (deterministic graph)     Agent (model-routed)
─────────────────────────────     ─────────────────────
Fixed node sequence               Model picks tools/path
Same path every run               Path varies by input
Cheaper, predictable              Flexible, costlier
```

| | **Workflow** | **Agent** |
|---|----------------|-----------|
| Routing | Code / rules | LLM (or hybrid) |
| Tools | Optional fixed nodes | Model-selected |
| Best for | ETL, pipelines, fixed SOPs | Open-ended Q&A, triage |

A **multi-agent system** is often a workflow graph where some nodes are agents (supervisor, workers) and others are deterministic (validation, logging).

In [ ]:
class NodeKind(Enum):
    WORKFLOW = "workflow"   # deterministic
    AGENT = "agent"         # model-routed


SHOPCO_GRAPH = [
    {"node": "validate_input", "kind": NodeKind.WORKFLOW, "desc": "Regex + length checks"},
    {"node": "supervisor", "kind": NodeKind.AGENT, "desc": "Routes to specialist worker"},
    {"node": "order_worker", "kind": NodeKind.AGENT, "desc": "lookup_order tool"},
    {"node": "policy_worker", "kind": NodeKind.AGENT, "desc": "search_policy tool"},
    {"node": "merge_response", "kind": NodeKind.WORKFLOW, "desc": "Format final answer"},
]

for n in SHOPCO_GRAPH:
    print(f"{n['node']:<18} [{n['kind'].value:<9}] {n['desc']}")

---

## 3. Architecture Layers — Where Components Live

```
┌─────────────────────────────────────────────────────────────┐
│  L7  Application API     FastAPI / gRPC / WebSocket         │
├─────────────────────────────────────────────────────────────┤
│  L6  Service layer       run_agent(), thread_id, auth       │
├─────────────────────────────────────────────────────────────┤
│  L5  Orchestration       LangGraph / AutoGen / CrewAI       │
├─────────────────────────────────────────────────────────────┤
│  L4  Agent loop          model node ↔ tool node (ReAct)     │
├─────────────────────────────────────────────────────────────┤
│  L3  Tool registry       schemas, risk tiers, HITL gates    │
├─────────────────────────────────────────────────────────────┤
│  L2  Integrations        DB, HTTP, search, code sandbox      │
├─────────────────────────────────────────────────────────────┤
│  L1  Observability       traces, metrics, evals             │
└─────────────────────────────────────────────────────────────┘
```

**Rule:** dependencies point **downward** — the agent loop (L4) should not import HTTP route handlers (L7). Keep layers separable so you can swap orchestration frameworks without rewriting integrations.

In [ ]:
ARCHITECTURE_LAYERS: dict[str, dict[str, str]] = {
    "L7": {
        "name": "Application API",
        "shopco_example": "POST /support/chat — HTTP entry point",
    },
    "L6": {
        "name": "Service layer",
        "shopco_example": "run_support_agent(question, thread_id, user_id)",
    },
    "L5": {
        "name": "Orchestration",
        "shopco_example": "Supervisor graph — route → worker → merge",
    },
    "L4": {
        "name": "Agent loop",
        "shopco_example": "Worker: decide → tool call → observation → answer",
    },
    "L3": {
        "name": "Tool registry",
        "shopco_example": "lookup_order, search_policy — schemas + allowlists",
    },
    "L2": {
        "name": "Integrations",
        "shopco_example": "In-memory ORDERS dict, POLICIES list (→ real DB/API)",
    },
    "L1": {
        "name": "Observability",
        "shopco_example": "TRACE_LOG — handoffs, tool calls, termination reason",
    },
}

for layer_id, info in ARCHITECTURE_LAYERS.items():
    print(f"{layer_id} {info['name']:<18} → {info['shopco_example']}")

---

## 4. Orchestrator vs Supervisor vs Worker

| Role | Responsibility | ShopCo example |
|------|----------------|----------------|
| **Orchestrator** | Runs the graph / loop mechanics | `ShopCoOrchestrator.run()` |
| **Supervisor** | Decides **which worker** acts next | Routes "ORD-1001" → order_worker |
| **Worker** | Specialist with narrow tools + prompt | order_worker has `lookup_order` only |

```
User ──► Supervisor ──► order_worker ──► back to Supervisor ──► END
              │
              └──► policy_worker ──► back to Supervisor
```

The **supervisor** is still an agent (or rule-based router), but its job is delegation — not doing every task itself.

---

## 5. Handoff and Termination Contracts

A **handoff** is not just "call another function." It is an explicit transfer of context with a contract:

- Source and destination agent ids
- Task summary (stable across turns)
- Constraints (read-only, cite sources)
- Expected output shape

**Termination** ends the loop. Common signals:

| Signal | Mechanism |
|--------|-----------|
| Task complete | Worker returns final text, no more tool calls |
| Max iterations | `step_count >= MAX_STEPS` |
| Guardrail | Blocked input or policy violation |
| Cost cap | Token budget exceeded |
| Human stop | HITL reject or user cancel |
| Error budget | N consecutive tool failures |

In [ ]:
@dataclass
class Handoff:
    handoff_id: str
    from_agent: str
    to_agent: str
    task: str
    constraints: list[str] = field(default_factory=list)
    expected_output: str = "concise answer with citations where applicable"
    context: dict[str, Any] = field(default_factory=dict)


class TerminationReason(Enum):
    FINAL_ANSWER = "final_answer"
    MAX_ITERATIONS = "max_iterations"
    GUARDRAIL_BLOCK = "guardrail_block"
    COST_CAP = "cost_cap"
    HUMAN_CANCEL = "human_cancel"
    ERROR_BUDGET = "error_budget"


@dataclass
class Termination:
    reason: TerminationReason
    detail: str = ""
    final_answer: str = ""


example_handoff = Handoff(
    handoff_id=str(uuid.uuid4())[:8],
    from_agent="supervisor",
    to_agent="order_worker",
    task="Look up order ORD-1001 and report status and amount",
    constraints=["read_only", "no billing changes"],
    expected_output="One paragraph with order_id, status, amount",
    context={"order_id": "ORD-1001"},
)

print("Handoff contract:")
print(pretty(example_handoff))

print("\nTermination reasons:", [r.value for r in TerminationReason])

---

## 6. Tools and Observations — The ACI Surface

Workers interact with the world through **tools**. Each tool call produces an **observation** the next agent turn reads.

```
Agent decides → tool_call(name, args) → execute → observation → Agent decides again
```

In production, observations should be structured (`[STATUS: ok] ...`) so the model can parse errors and retry.

In [ ]:
TRACE_LOG: list[dict[str, Any]] = []


def trace(event: str, **kwargs: Any) -> None:
    TRACE_LOG.append({
        "ts": datetime.now(timezone.utc).strftime("%H:%M:%S"),
        "event": event,
        **kwargs,
    })


def lookup_order(order_id: str) -> str:
    oid = order_id.upper()
    if oid not in ORDERS:
        return f"[STATUS: error] Unknown order {oid}"
    return f"[STATUS: ok] {json.dumps({'order_id': oid, **ORDERS[oid]})}"


def search_policy(query: str) -> str:
    q = query.lower()
    hits = [p for p in POLICIES if any(t in p["text"].lower() for t in q.split())]
    if not hits:
        hits = POLICIES
    text = " | ".join(f"[{h['id']}] {h['text']}" for h in hits)
    return f"[STATUS: ok] {text}"


TOOL_REGISTRY: dict[str, Callable[..., str]] = {
    "lookup_order": lookup_order,
    "search_policy": search_policy,
}

WORKER_TOOLS: dict[str, list[str]] = {
    "order_worker": ["lookup_order"],
    "policy_worker": ["search_policy"],
}

# Demo: tool → observation
obs = lookup_order("ORD-1001")
trace("tool_call", agent="order_worker", tool="lookup_order", observation=obs[:60])
print("Observation:", obs)
print("Trace entry:", TRACE_LOG[-1])

---

## 7. Working Supervisor + Workers — ShopCo MAS

Below is a minimal but complete multi-agent system:

1. **Supervisor** routes by keywords (offline stand-in for an LLM router).
2. **Workers** execute one tool each and format an answer.
3. **Orchestrator** manages handoffs, step limits, and termination.
4. **Trace log** (L1) records every event.

In [ ]:
MAX_STEPS = 6


def supervisor_route(user_query: str) -> str:
    """Rule-based supervisor — production would use an LLM with structured output."""
    q = user_query.lower()
    if "ord-" in q or "order" in q and any(c.isdigit() for c in q):
        return "order_worker"
    if any(w in q for w in ("return", "cancel", "policy", "refund")):
        return "policy_worker"
    return "policy_worker"  # default for general questions


def run_worker(worker_id: str, handoff: Handoff) -> tuple[str, str]:
    """Worker agent loop: one tool call → observation → formatted answer."""
    allowed = WORKER_TOOLS.get(worker_id, [])

    if worker_id == "order_worker":
        order_id = handoff.context.get("order_id", "ORD-1001")
        import re
        match = re.search(r"ORD-\d{4}", handoff.task.upper())
        if match:
            order_id = match.group(0)
        tool = "lookup_order"
        obs = TOOL_REGISTRY[tool](order_id=order_id)
        trace("tool_call", agent=worker_id, tool=tool, args={"order_id": order_id})
        if "[STATUS: error]" in obs:
            return obs, ""
        data = json.loads(obs.replace("[STATUS: ok] ", ""))
        answer = (
            f"Order {data['order_id']} is {data['status']} "
            f"(${data['amount']:.2f}) for {data['customer']}."
        )
        return obs, answer

    if worker_id == "policy_worker":
        tool = "search_policy"
        query = handoff.task
        obs = TOOL_REGISTRY[tool](query=query)
        trace("tool_call", agent=worker_id, tool=tool, args={"query": query[:40]})
        policy_text = obs.replace("[STATUS: ok] ", "")
        answer = f"Per ShopCo policy: {policy_text}"
        return obs, answer

    return "[STATUS: error] Unknown worker", ""


@dataclass
class MasRunResult:
    answer: str
    termination: Termination
    handoffs: list[Handoff]
    steps: int


class ShopCoOrchestrator:
    """L5/L6 orchestrator — runs supervisor → worker → termination."""

    def __init__(self, max_steps: int = MAX_STEPS):
        self.max_steps = max_steps

    def run(self, user_query: str, thread_id: str = "t-001") -> MasRunResult:
        TRACE_LOG.clear()
        trace("session_start", thread_id=thread_id, query=user_query[:80])

        handoffs: list[Handoff] = []
        steps = 0

        # Step 1: Supervisor routes
        steps += 1
        worker_id = supervisor_route(user_query)
        trace("supervisor_route", next_worker=worker_id)

        if steps >= self.max_steps:
            return MasRunResult(
                "", Termination(TerminationReason.MAX_ITERATIONS, "exceeded before worker"), handoffs, steps
            )

        # Step 2: Handoff to worker
        handoff = Handoff(
            handoff_id=str(uuid.uuid4())[:8],
            from_agent="supervisor",
            to_agent=worker_id,
            task=user_query,
            constraints=["read_only"],
            context={"thread_id": thread_id},
        )
        handoffs.append(handoff)
        trace("handoff", **{"from": handoff.from_agent, "to": handoff.to_agent, "id": handoff.handoff_id})

        # Step 3: Worker executes
        steps += 1
        observation, answer = run_worker(worker_id, handoff)
        trace("observation", agent=worker_id, preview=observation[:60])

        if "[STATUS: error]" in observation:
            return MasRunResult(
                observation,
                Termination(TerminationReason.ERROR_BUDGET, observation),
                handoffs,
                steps,
            )

        # Step 4: Terminate with final answer
        trace("termination", reason="final_answer", steps=steps)
        return MasRunResult(
            answer,
            Termination(TerminationReason.FINAL_ANSWER, final_answer=answer),
            handoffs,
            steps,
        )


orchestrator = ShopCoOrchestrator()

for query in [
    "Where is order ORD-1001?",
    "What is your return policy?",
    "Can I cancel order ORD-1002?",
]:
    result = orchestrator.run(query)
    print(f"\nQ: {query}")
    print(f"A: {result.answer}")
    print(f"   worker={result.handoffs[-1].to_agent} | steps={result.steps} | reason={result.termination.reason.value}")

---

## 8. Inspect the Trace — L1 Observability

Every handoff, tool call, and termination should be traceable. This is the **observation layer** for debugging and evals.

In [ ]:
_ = orchestrator.run("Where is order ORD-1001?")

print(f"{'Time':<10} {'Event':<18} Details")
print("-" * 70)
for entry in TRACE_LOG:
    details = {k: v for k, v in entry.items() if k not in ("ts", "event")}
    print(f"{entry['ts']:<10} {entry['event']:<18} {details}")

---

## 9. Map Vocabulary to LangGraph

| MAS term | LangGraph construct |
|----------|---------------------|
| Agent loop | `StateGraph` with model node + `ToolNode` |
| Orchestrator | Compiled graph + `invoke` / `astream` |
| Supervisor | Parent graph routing to worker subgraphs |
| Handoff | State field `next_agent` or dedicated routing edge |
| Termination | `END` edge, conditional router, `recursion_limit` |
| Observation | `ToolMessage.content` |
| Checkpoint | `checkpointer=` on `compile()` for durable state |

LangGraph makes the **orchestrator** explicit as a graph. Workers are often **subgraphs** with their own tool sets.

---

## 10. Map Vocabulary to AutoGen and CrewAI

| MAS term | AutoGen | CrewAI |
|----------|---------|--------|
| Agent | `AssistantAgent` | `Agent` with role + goal |
| Tool | `register_for_llm` / function | `@tool` decorator |
| Supervisor | `GroupChatManager` | hierarchical `Process` |
| Handoff | speaker selection / message | task delegation between agents |
| Termination | `is_termination_msg` | `max_iter` on crew |
| Observation | function return value | task output string |

Frameworks differ in **API shape** — the **vocabulary** stays stable. When you switch frameworks, translate concepts, not just syntax.

---

## 11. Map Vocabulary to MCP (Model Context Protocol)

| MAS term | MCP equivalent |
|----------|----------------|
| Tool | MCP **tool** exposed by a server |
| Observation | Tool **result** content blocks |
| Registry | MCP server capability / tool list |
| Least privilege | Per-server connection scopes |

MCP standardizes **tool discovery and transport** across processes — it does not replace your agent loop or supervisor. Think of MCP as the **L2/L3 wire format** for tools hosted outside your app.

In [ ]:
# Conceptual MCP-style tool manifest (not a live MCP server)
MCP_TOOL_MANIFEST = [
    {
        "name": "lookup_order",
        "description": "Fetch order status by ORD-#### id",
        "inputSchema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
    {
        "name": "search_policy",
        "description": "Search ShopCo return and cancellation policies",
        "inputSchema": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
]

print("MCP-style tool manifest:")
for tool in MCP_TOOL_MANIFEST:
    print(f"  • {tool['name']}: {tool['description']}")

---

## 12. Single-Agent vs Multi-Agent — Vocabulary View

| | Single-agent | Multi-agent |
|---|--------------|-------------|
| Tool selection | One model picks from full tool menu | Supervisor or handoffs split tools by role |
| State | One message thread | Shared graph state or per-agent threads |
| Coordination cost | Lower | Higher — handoffs, merge logic |
| When to use | < 8 tools, one domain | Conflicting specialties, parallel review |

Multi-agent is not automatically better. Use it when **role separation** or **parallel specialists** justify the coordination overhead.

In [ ]:
SINGLE_AGENT_TOOLS = ["lookup_order", "search_policy", "update_billing", "cancel_order"]
MULTI_AGENT_LAYOUT = {
    "supervisor": [],
    "order_worker": ["lookup_order"],
    "policy_worker": ["search_policy"],
    "billing_worker": ["update_billing", "cancel_order"],
}

print("Single-agent tool surface:", len(SINGLE_AGENT_TOOLS), "tools on one agent")
print("Multi-agent tool surface:")
for agent, tools in MULTI_AGENT_LAYOUT.items():
    print(f"  {agent:<16} {tools or '(routes only)'}")

---

## 13. End-to-End Vocabulary Flow

```
User query
    │
    ▼
[L7] API ──► [L6] Service ──► [L5] Orchestrator
                                    │
                    ┌───────────────┼───────────────┐
                    ▼               ▼               ▼
              Supervisor      handoff         termination
                    │               │               │
                    ▼               ▼               ▼
              Worker agent    Tool calls      Observation
                    │               │               │
                    └───────────────┴───────────────┘
                                    │
                                    ▼
                              [L1] Trace log
```

Use this diagram in design reviews — everyone should point at the same boxes when discussing bugs or new features.

---

## 14. Framework Selection (High Level)

| Need | Start with |
|------|------------|
| Graph control, checkpoints, HITL interrupts | **LangGraph** |
| Conversational multi-agent chat | **AutoGen** |
| Role-based task crews | **CrewAI** |
| Standard tool servers across repos | **MCP** |
| Minimal dependencies | Provider SDK + custom loop (like this notebook) |

No framework removes the need for good **tool schemas**, **security gates**, and **observability**. Vocabulary first; framework second.

---

## 15. Exercise — Label Your System

Take a current or planned project and map components to layers:

| Layer | Your component |
|-------|----------------|
| L7 API | ? |
| L6 Service | ? |
| L5 Orchestrator | ? |
| L4 Agent loop | ? |
| L3 Tool registry | ? |
| L2 Integrations | ? |
| L1 Observability | ? |

Any blank row is a gap to close before production. Also name your **supervisor**, **workers**, and **termination conditions**.

In [ ]:
LABEL_YOUR_SYSTEM = {
    "L7": "",  # e.g. POST /api/chat
    "L6": "",  # e.g. run_agent(thread_id)
    "L5": "",  # e.g. LangGraph StateGraph
    "L4": "",  # e.g. model ↔ ToolNode loop
    "L3": "",  # e.g. tool schemas + HITL
    "L2": "",  # e.g. Postgres, Stripe API
    "L1": "",  # e.g. Langfuse traces
}

filled = sum(1 for v in LABEL_YOUR_SYSTEM.values() if v.strip())
print(f"Layers documented: {filled}/{len(LABEL_YOUR_SYSTEM)}")
if filled == 0:
    print("Fill in LABEL_YOUR_SYSTEM above for your project.")

---

## 16. Quick Reference Card

| Term | One-line definition |
|------|---------------------|
| **Agent** | LLM that perceives, decides, and acts in a loop |
| **Tool** | Schema-defined capability the model invokes |
| **Observation** | Tool output the model reads on the next turn |
| **Orchestrator** | Code that runs the graph / agent loop |
| **Supervisor** | Agent that delegates to specialist workers |
| **Handoff** | Explicit context transfer between agents |
| **Termination** | Why and how the loop stopped |

Pin this table in your team wiki — it prevents "agent" and "orchestrator" from being used interchangeably.

---

## 17. Check Your Understanding

1. What is the difference between an **orchestrator** and a **supervisor**?
2. What must a **handoff** message include beyond "call worker B"?
3. Name three **termination** conditions besides "task complete."
4. Which architecture layer owns tool schemas and risk tiers?
5. How does an **observation** differ from a **tool**?
6. When would you choose multi-agent over single-agent?
7. What does MCP standardize — and what does it *not* replace?

---

## 18. Summary

Multi-agent systems need a **shared vocabulary** before framework debates:

- **Agents** perceive, decide, and act; **tools** are their ACI; **observations** feed the next turn.
- **Orchestrators** run the loop; **supervisors** delegate; **workers** specialize.
- **Handoffs** carry explicit contracts; **termination** ends the loop for a known reason.
- Architecture **layers** (L7 API → L1 observability) keep concerns separable.
- Frameworks (LangGraph, AutoGen, CrewAI, MCP) map to the same terms — learn vocabulary once.

The ShopCo demo showed every term in code: supervisor routing, worker tools, handoff records, trace logs, and clean termination. That is the blueprint for every MAS you build next.